In [1]:
import json
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/gdrive', force_remount=True)

# Create root directory
root_dir = '/content/gdrive/My Drive/StudentSyllabusBot'
os.makedirs(root_dir, exist_ok=True)

data_dir = os.path.join(root_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

print("✅ Directories created!")
print(f"📁 Data will be saved to: {data_dir}")

Mounted at /content/gdrive
✅ Directories created!
📁 Data will be saved to: /content/gdrive/My Drive/StudentSyllabusBot/data


In [14]:
import pdfplumber, re, json, os
from google.colab import files, drive

# === MOUNT DRIVE ===
drive.mount('/content/gdrive', force_remount=True)
root_dir = '/content/gdrive/My Drive/StudentSyllabusBot'
data_dir = os.path.join(root_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

# === UPLOAD PDF ===
print("⬆️ Upload your syllabus PDF now:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# === EXTRACT LINES WITH FONT SIZE ===
all_lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        words = page.extract_words(extra_attrs=["size"])
        line_map = {}
        for w in words:
            key = round(w["top"], 1)
            if key not in line_map:
                line_map[key] = []
            line_map[key].append(w)
        for _, line_words in sorted(line_map.items()):
            text = " ".join([w["text"] for w in line_words]).strip()
            if not text:
                continue
            avg_size = sum([w["size"] for w in line_words]) / len(line_words)
            all_lines.append({"text": text, "size": avg_size})

# === FIND COURSE BLOCKS ===
documents = []
doc_id = 1

# indices of lines with "Course Code"
course_code_indices = [i for i, ln in enumerate(all_lines) if re.search(r'Course Code\s+\d{2}[A-Z]{2,3}\d{2,3}', ln["text"])]

for idx, start_i in enumerate(course_code_indices):
    end_i = course_code_indices[idx+1] if idx+1 < len(course_code_indices) else len(all_lines)
    block_lines = all_lines[start_i:end_i]

    # Extract course code
    code_match = re.search(r'Course Code\s+(\d{2}[A-Z]{2,3}\d{2,3})', block_lines[0]["text"])
    course_code = code_match.group(1) if code_match else "UNKNOWN"

    # Find course title = largest font ABOVE course code within last 5 lines
    title = "Unknown Course"
    search_start = max(0, start_i - 8)
    candidate_lines = all_lines[search_start:start_i]
    if candidate_lines:
        best = max(candidate_lines, key=lambda x: x["size"])
        title = best["text"].strip()

    # Combine block text
    block_text = "\n".join([ln["text"] for ln in block_lines])

    # === Extract Objectives ===
    obj_match = re.search(r'Course Objectives?:\s*(.*?)(?=Module\s*[–-]|Course Outcomes|Textbooks|References|$)', block_text, re.DOTALL)
    if obj_match:
        documents.append({
            "id": doc_id,
            "course_code": course_code,
            "course_title": title,
            "section": "Course Objectives",
            "content": obj_match.group(1).strip()
        })
        doc_id += 1

    # === Extract Modules ===
    module_matches = re.finditer(
        r'(Module\s*[–-]\s*\d+)\s*(.*?)(?=Module\s*[–-]\s*\d+|Course Outcomes|Textbooks|References|$)',
        block_text,
        re.DOTALL
    )
    for m in module_matches:
        module_title = m.group(1).strip()
        module_content = m.group(2).strip()
        documents.append({
            "id": doc_id,
            "course_code": course_code,
            "course_title": title,
            "section": module_title,
            "content": module_content
        })
        doc_id += 1

    # === Extract Outcomes ===
    out_match = re.search(r'Course Outcomes?:\s*(.*?)(?=Textbooks|References|$)', block_text, re.DOTALL)
    if out_match:
        documents.append({
            "id": doc_id,
            "course_code": course_code,
            "course_title": title,
            "section": "Course Outcomes",
            "content": out_match.group(1).strip()
        })
        doc_id += 1

    # === Extract Textbooks ===
    tb_match = re.search(r'Textbooks?:\s*(.*?)(?=References|$)', block_text, re.DOTALL)
    if tb_match:
        documents.append({
            "id": doc_id,
            "course_code": course_code,
            "course_title": title,
            "section": "Textbooks",
            "content": tb_match.group(1).strip()
        })
        doc_id += 1

    # === Extract References ===
    ref_match = re.search(r'References?:\s*(.*?)(?=$)', block_text, re.DOTALL)
    if ref_match:
        documents.append({
            "id": doc_id,
            "course_code": course_code,
            "course_title": title,
            "section": "References",
            "content": ref_match.group(1).strip()
        })
        doc_id += 1

# === SAVE JSONL ===
jsonl_path = os.path.join(data_dir, "syllabus_chunks.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print(f"✅ Created {len(documents)} chunks")
print(f"✅ JSONL saved to: {jsonl_path}")
print("\n📄 Sample:")
print(documents[0])

Mounted at /content/gdrive
⬆️ Upload your syllabus PDF now:


Saving 6aiml.pdf to 6aiml (2).pdf
✅ Created 103 chunks
✅ JSONL saved to: /content/gdrive/My Drive/StudentSyllabusBot/data/syllabus_chunks.jsonl

📄 Sample:
{'id': 1, 'course_code': '21HSS61', 'course_title': 'VI Semester Syllabus', 'section': 'Course Objectives', 'content': 'This course will enable students to:\n1. Define the fundamentals of Project Management.\n2. Identify the strategies involved in selection, prioritization, planning & scheduling of a project.\n3. Understand the time value of money & apply it for decision making.\n4. Analyze project risk, progress & results.\n5. Make awareness about various sources of finance.\nGain Knowledge on working capital & capital budgeting.\n6.'}


In [15]:
# === STEP 9: Install dependencies ===
!pip -q install sentence-transformers faiss-cpu

import json, os, pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# === STEP 10: Load JSONL chunks ===
jsonl_path = os.path.join(data_dir, "syllabus_chunks.jsonl")

documents = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            documents.append(json.loads(line))

print(f"✅ Loaded {len(documents)} chunks")

# === STEP 11: Load embedding model ===
model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)

# === STEP 12: Create embeddings ===
texts = [doc["content"] for doc in documents]
embeddings = embedding_model.encode(texts, convert_to_numpy=True)

# === STEP 13: Build FAISS index ===
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

# === STEP 14: Save FAISS + metadata ===
faiss_path = os.path.join(data_dir, "syllabus_index.faiss")
faiss.write_index(index, faiss_path)

metadata_path = os.path.join(data_dir, "syllabus_metadata.pkl")
with open(metadata_path, "wb") as f:
    pickle.dump(documents, f)

print(f"✅ FAISS index saved to: {faiss_path}")
print(f"✅ Metadata saved to: {metadata_path}")

✅ Loaded 103 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index saved to: /content/gdrive/My Drive/StudentSyllabusBot/data/syllabus_index.faiss
✅ Metadata saved to: /content/gdrive/My Drive/StudentSyllabusBot/data/syllabus_metadata.pkl


In [16]:
# ===== INSTALL ALL REQUIRED PACKAGES =====
!pip -q install torch transformers sentence-transformers faiss-cpu gradio pdfplumber

# ===== IMPORT EVERYTHING =====
import torch
import faiss
import gradio as gr
import pdfplumber
import numpy as np
import json, os, re, pickle
from google.colab import drive, files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("✅ All dependencies installed and imported successfully!")

✅ All dependencies installed and imported successfully!


In [21]:
# === TinyLlama 1.1B Chat (clean answers after retrieval) ===
!pip -q install transformers accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Save model to Drive cache
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
cache_dir = os.path.join(root_dir, "models", "tinyllama-1.1b")
os.makedirs(cache_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    torch_dtype=torch.float16,
    device_map="auto"
)

def answer_question(query):
    retrieved = retrieve(query, k=3)

    # Build context
    context = ""
    for doc in retrieved:
        context += f"\n[{doc['course_title']} - {doc['section']}]\n{doc['content']}\n"

    # Prompt for clean student-friendly answer
    prompt = f"""
You are a helpful assistant for students.
Answer clearly in short bullet points using the syllabus context only.

Question: {query}

Context:
{context}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Answer:")[-1].strip()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

this uses tiny llama 1.1b model to giev human readable answers


In [22]:
import gradio as gr

def chatbot_ui(query):
    if not query.strip():
        return "Please ask a question about your syllabus."
    return answer_question(query)

demo = gr.Interface(
    fn=chatbot_ui,
    inputs=gr.Textbox(lines=2, placeholder="Ask about your syllabus..."),
    outputs="markdown",
    title="🎓 Student Syllabus Chatbot",
    description="Ask any question about your syllabus. Answers are generated using TinyLlama + your syllabus context."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0e63d169bfbaa43c6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


this is just retrieval and no use of any llm model in chatbot


In [19]:
# === STEP 15: Install Gradio & Transformers ===
!pip -q install gradio transformers torch

import faiss, pickle, os
import numpy as np
from sentence_transformers import SentenceTransformer
import gradio as gr
from transformers import pipeline

# === STEP 16: Load FAISS + metadata ===
faiss_path = os.path.join(data_dir, "syllabus_index.faiss")
metadata_path = os.path.join(data_dir, "syllabus_metadata.pkl")

index = faiss.read_index(faiss_path)
with open(metadata_path, "rb") as f:
    documents = pickle.load(f)

# === STEP 17: Load embedding model ===
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# === STEP 18: Load lightweight LLM ===
generator = pipeline("text-generation", model="distilgpt2", device=0 if torch.cuda.is_available() else -1)

# === STEP 19: Retriever ===
def retrieve(query, k=3):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_emb.astype(np.float32), k)
    results = [documents[i] for i in indices[0] if i < len(documents)]
    return results

# === STEP 20: Response Generator ===
def answer_question(query):
    retrieved = retrieve(query, k=3)

    # Build context
    context = ""
    for doc in retrieved:
        context += f"\n[{doc['course_title']} - {doc['section']}]\n{doc['content']}\n"

    # Simple response
    response = f"📚 **Answer based on your syllabus:**\n\n{context}\n\n✅ *This is pulled from your syllabus.*"
    return response

# === STEP 21: Gradio UI ===
def chatbot_ui(query):
    if not query.strip():
        return "Please ask a question about your syllabus."
    return answer_question(query)

demo = gr.Interface(
    fn=chatbot_ui,
    inputs=gr.Textbox(lines=2, placeholder="Ask about your syllabus..."),
    outputs="markdown",
    title="🎓 Student Syllabus Chatbot",
    description="Ask any question about your syllabus. The chatbot will retrieve the relevant module or section."
)

demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7bbd975db71a0383e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
